# MPECSS - Configurable Benchmark Template

This notebook runs any of the three benchmark suites from a single template.

- **Choose dataset**: Set `DATASET` below to `mpeclib`, `macmpec`, or `nosbench`
- **Resume support**: Built in via `kaggle_setup/resumable_benchmark.py`
- **Repo source**: GitHub (clones fresh by default)

Run the cells in order. After a Kaggle restart, rerun the repository and
install cells, then run the **Resume** cell.


## 1. Configuration

Change `DATASET` to choose which benchmark suite to run.


In [ ]:
# ┌─────────────────────────────────────────────────┐
# │  CHANGE THIS to run a different benchmark suite  │
# └─────────────────────────────────────────────────┘
DATASET = 'mpeclib'  # Options: 'mpeclib', 'macmpec', 'nosbench'
RUN_TAG = f'Kaggle_{DATASET.title()}'

WORKERS = 4  # Match Kaggle's 4 CPU cores
TIMEOUT = 3600
NUM_PROBLEMS = None
PROBLEM_FILTER = ""
MEM_LIMIT_GB = None
SAVE_LOGS = True
SORT_BY_SIZE = False
SHUFFLE = True
RETRY_FAILED_ON_RESUME = False
DATASET_PATH_OVERRIDE = ''

# Repository source - always clone fresh from GitHub
REPO_DIR = "/kaggle/working/Org-MPECSS"
REPO_GIT_URL = "https://github.com/mrsaurabhtanwar/Org-MPECSS.git"


## 2. Prepare The Repository


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)

# Always clone fresh from GitHub to get latest code
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print(f"Cloning repo from Git: {REPO_GIT_URL}")
subprocess.run(["git", "clone", REPO_GIT_URL, str(REPO_DIR)], check=True)

# Add to Python path
sys.path.insert(0, str(REPO_DIR))

print(f"Repository ready at: {REPO_DIR}")


## 3. Install Dependencies


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Org-MPECSS

python -m pip install -q --upgrade pip
python -m pip install -q -e .

echo "[OK] Editable install completed"


## 4. Inspect The Kaggle Instance


In [ ]:
import multiprocessing
import platform
import psutil

print("=" * 60)
print("System Information")
print("=" * 60)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"Logical CPU cores: {multiprocessing.cpu_count()}")

mem = psutil.virtual_memory()
print(f"Total RAM: {mem.total / 1024**3:.1f} GB")
print(f"Available RAM: {mem.available / 1024**3:.1f} GB")
print("=" * 60)


## 5. Run Preflight Checks


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Org-MPECSS
python scripts/preflight_checks.py


## 6. Load The Runner Helper


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)
WRAPPER = REPO_DIR / "kaggle_setup" / "resumable_benchmark.py"

def run_benchmark(resume_latest=False, summary_only=False):
    command = [
        sys.executable,
        str(WRAPPER),
        "--dataset",
        DATASET,
        "--repo-dir",
        str(REPO_DIR),
        "--tag",
        RUN_TAG,
        "--workers",
        str(WORKERS),
        "--timeout",
        str(TIMEOUT),
        "--skip-preflight",
    ]

    if SAVE_LOGS:
        command.append("--save-logs")
    if SORT_BY_SIZE:
        command.append("--sort-by-size")
    if SHUFFLE:
        command.append("--shuffle")
    else:
        command.append("--no-shuffle")
    if PROBLEM_FILTER:
        command.extend(["--problem", PROBLEM_FILTER])
    if NUM_PROBLEMS is not None:
        command.extend(["--num-problems", str(NUM_PROBLEMS)])
    if MEM_LIMIT_GB is not None:
        command.extend(["--mem-limit-gb", str(MEM_LIMIT_GB)])
    if DATASET_PATH_OVERRIDE:
        command.extend(["--path", DATASET_PATH_OVERRIDE])
    if RETRY_FAILED_ON_RESUME:
        command.append("--retry-failed")
    if resume_latest:
        command.append("--resume-latest")
    if summary_only:
        command.append("--summary-only")

    print("+", " ".join(command))
    subprocess.run(command, check=True)


## 7. Launch A Fresh Run


In [ ]:
run_benchmark()


## 8. Resume After A Kaggle Restart


In [ ]:
run_benchmark(resume_latest=True)


## 9. Progress Summary


In [ ]:
run_benchmark(summary_only=True)


## 10. Copy Results For Download


In [ ]:
%%bash
set -euo pipefail
mkdir -p /kaggle/working/outputs
cp -r /kaggle/working/Org-MPECSS/results/* /kaggle/working/outputs/ || true

echo "[OK] Results copied to /kaggle/working/outputs/"
echo "Download from the Kaggle file browser or output panel"


## 11. Software Versions For Your Paper


In [ ]:
import casadi
import matplotlib
import numpy
import pandas
import platform
import psutil
import scipy

print("Software Versions")
print("=" * 40)
print(f"Python: {platform.python_version()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"CasADi: {casadi.__version__}")
print(f"psutil: {psutil.__version__}")
print("=" * 40)
